# Ensemble Learning for Regression - California Housing Dataset

## Overview

Ensemble learning is a machine learning paradigm where multiple models (often called "weak learners") are strategically combined to solve a particular computational intelligence problem. Ensemble methods can significantly improve the performance of machine learning models.

**Ensemble Techniques Covered:**
1. **Bagging (Bootstrap Aggregating)**: Training multiple models on different random subsets of data
2. **Voting (Hard and Soft)**: Combining predictions from multiple models
3. **Stacking (Stacked Generalization)**: Using predictions from multiple models as input to a meta-model
4. **Blending**: Similar to stacking but uses a hold-out validation set

**Benefits of Ensemble Learning:**
- **Improved Accuracy**: Combining multiple models often outperforms individual models
- **Reduced Overfitting**: Averaging predictions reduces variance
- **Better Generalization**: Different models capture different patterns in data
- **Robustness**: Less sensitive to noise and outliers

## Step 1: Import Required Libraries

We need to import all necessary libraries for data manipulation, visualization, and ensemble learning algorithms.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Sklearn modules
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Base models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# Ensemble methods
from sklearn.ensemble import (
    BaggingRegressor,
    VotingRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor
)

# Try to import XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("XGBoost imported successfully!")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")

## Step 2: Load and Prepare the Dataset

We'll use the California Housing dataset for consistency with other regression notebooks.

In [ ]:
# Load the dataset
housing = fetch_california_housing()

# Create DataFrame
df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

# Add target variable
df["Price"] = housing.target

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nDataset Statistics:")
display(df.describe())

## Step 3: Data Preprocessing

Standard preprocessing steps: check for missing values, split features and target, scale features, and create train-test split.

In [ ]:
# Check for missing values and duplicates
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

# Split features and target
X = df.drop("Price", axis=1)
y = df["Price"]

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)

## Step 4: Bagging (Bootstrap Aggregating)

**Bagging Concept:**

Bagging (Bootstrap Aggregating) is an ensemble technique that trains multiple models on different random subsets of the training data (with replacement). The final prediction is obtained by averaging the predictions from all models.

**Bagging Formula:**

$$\hat{y} = \frac{1}{N} \sum_{i=1}^{N} f_i(x)$$

Where:
- **ŷ** = final prediction
- **N** = number of base models
- **fᵢ(x)** = prediction from the i-th model
- **x** = input features

**Key Parameters:**
- **n_estimators**: Number of base models to train
- **max_samples**: Number of samples to draw from X for each base model
- **max_features**: Number of features to draw from X for each base model
- **bootstrap**: Whether samples are drawn with replacement (default=True)

**Advantages:**
- **Reduces Variance**: Averaging predictions reduces overfitting
- **Parallelizable**: Models can be trained independently
- **Robust**: Less sensitive to noise and outliers

**Disadvantages:**
- **Computationally Expensive**: Training multiple models
- **Memory Intensive**: Storing multiple models
- **Less Interpretability**: Harder to understand than single models

In [ ]:
# Bagging with Decision Tree as base estimator
bagging_dt = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training Bagging Regressor with Decision Tree...")
bagging_dt.fit(X_train, y_train)
y_pred_bagging_dt = bagging_dt.predict(X_test)

# Evaluate
mae_bagging_dt = mean_absolute_error(y_test, y_pred_bagging_dt)
mse_bagging_dt = mean_squared_error(y_test, y_pred_bagging_dt)
rmse_bagging_dt = np.sqrt(mse_bagging_dt)
r2_bagging_dt = r2_score(y_test, y_pred_bagging_dt)

print("\nBagging (Decision Tree) Results:")
print(f"MAE: {mae_bagging_dt:.4f}")
print(f"RMSE: {rmse_bagging_dt:.4f}")
print(f"R²: {r2_bagging_dt:.4f}")

In [ ]:
# Bagging with Linear Regression as base estimator
bagging_lr = BaggingRegressor(
    estimator=LinearRegression(),
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training Bagging Regressor with Linear Regression...")
bagging_lr.fit(X_train, y_train)
y_pred_bagging_lr = bagging_lr.predict(X_test)

# Evaluate
mae_bagging_lr = mean_absolute_error(y_test, y_pred_bagging_lr)
mse_bagging_lr = mean_squared_error(y_test, y_pred_bagging_lr)
rmse_bagging_lr = np.sqrt(mse_bagging_lr)
r2_bagging_lr = r2_score(y_test, y_pred_bagging_lr)

print("\nBagging (Linear Regression) Results:")
print(f"MAE: {mae_bagging_lr:.4f}")
print(f"RMSE: {rmse_bagging_lr:.4f}")
print(f"R²: {r2_bagging_lr:.4f}")

## Step 5: Voting Regressor

**Voting Concept:**

Voting is an ensemble technique that combines predictions from multiple different models. There are two types of voting:

1. **Hard Voting**: Each model votes for a prediction, and the final prediction is the average of all votes
2. **Soft Voting**: Each model provides a weighted prediction based on its confidence

**Voting Formula (for Regression):**

$$\hat{y} = \sum_{i=1}^{N} w_i \cdot f_i(x)$$

Where:
- **ŷ** = final prediction
- **N** = number of base models
- **wᵢ** = weight for the i-th model (default = 1/N)
- **fᵢ(x)** = prediction from the i-th model
- **x** = input features

**Key Parameters:**
- **estimators**: List of (name, model) tuples
- **weights**: Optional weights for each model

**Advantages:**
- **Diverse Models**: Combines different types of algorithms
- **Improved Accuracy**: Often outperforms individual models
- **Flexible**: Can use any regression models

**Disadvantages:**
- **Computationally Expensive**: Training multiple models
- **Complex**: Requires careful selection of base models
- **Less Interpretability**: Harder to understand than single models

In [ ]:
# Define base models for voting
estimators = [
    ('linear', LinearRegression()),
    ('ridge', Ridge(alpha=1.0)),
    ('lasso', Lasso(alpha=1.0)),
    ('decision_tree', DecisionTreeRegressor(max_depth=5, random_state=42)),
    ('knn', KNeighborsRegressor(n_neighbors=5))
]

# Voting Regressor
voting_reg = VotingRegressor(
    estimators=estimators,
    n_jobs=-1
)

print("Training Voting Regressor...")
voting_reg.fit(X_train, y_train)
y_pred_voting = voting_reg.predict(X_test)

# Evaluate
mae_voting = mean_absolute_error(y_test, y_pred_voting)
mse_voting = mean_squared_error(y_test, y_pred_voting)
rmse_voting = np.sqrt(mse_voting)
r2_voting = r2_score(y_test, y_pred_voting)

print("\nVoting Regressor Results:")
print(f"MAE: {mae_voting:.4f}")
print(f"RMSE: {rmse_voting:.4f}")
print(f"R²: {r2_voting:.4f}")

In [ ]:
# Voting Regressor with weights (giving more importance to better models)
weights = [1, 1, 1, 2, 1]  # Give more weight to Decision Tree

voting_reg_weighted = VotingRegressor(
    estimators=estimators,
    weights=weights,
    n_jobs=-1
)

print("Training Weighted Voting Regressor...")
voting_reg_weighted.fit(X_train, y_train)
y_pred_voting_weighted = voting_reg_weighted.predict(X_test)

# Evaluate
mae_voting_weighted = mean_absolute_error(y_test, y_pred_voting_weighted)
mse_voting_weighted = mean_squared_error(y_test, y_pred_voting_weighted)
rmse_voting_weighted = np.sqrt(mse_voting_weighted)
r2_voting_weighted = r2_score(y_test, y_pred_voting_weighted)

print("\nWeighted Voting Regressor Results:")
print(f"MAE: {mae_voting_weighted:.4f}")
print(f"RMSE: {rmse_voting_weighted:.4f}")
print(f"R²: {r2_voting_weighted:.4f}")

## Step 6: Stacking (Stacked Generalization)

**Stacking Concept:**

Stacking (Stacked Generalization) is an ensemble technique that combines multiple models by training a meta-model (also called a blender or meta-learner) on the predictions of the base models. The meta-model learns how to best combine the predictions from the base models.

**Stacking Process:**

1. **Split Data**: Divide training data into k-folds
2. **Train Base Models**: Train each base model on k-1 folds
3. **Generate Predictions**: Use each base model to predict on the held-out fold
4. **Create Meta-Features**: Use these predictions as features for the meta-model
5. **Train Meta-Model**: Train meta-model on the meta-features
6. **Final Prediction**: Use base models to predict on test data, then meta-model to combine

**Stacking Formula:**

$$\hat{y} = g(f_1(x), f_2(x), ..., f_N(x))$$

Where:
- **ŷ** = final prediction
- **g()** = meta-model function
- **fᵢ(x)** = prediction from the i-th base model
- **x** = input features

**Key Parameters:**
- **estimators**: List of (name, model) tuples for base models
- **final_estimator**: Meta-model to combine predictions (default=LinearRegression)
- **cv**: Cross-validation strategy for generating meta-features

**Advantages:**
- **High Accuracy**: Often achieves state-of-the-art performance
- **Flexible**: Can use any models as base learners
- **Optimal Combination**: Meta-model learns optimal combination

**Disadvantages:**
- **Computationally Expensive**: Training multiple models + meta-model
- **Complex**: More complex to implement and tune
- **Risk of Overfitting**: Meta-model can overfit if not properly regularized

In [ ]:
from sklearn.ensemble import StackingRegressor

# Define base models for stacking
base_estimators = [
    ('linear', LinearRegression()),
    ('ridge', Ridge(alpha=1.0)),
    ('lasso', Lasso(alpha=1.0)),
    ('decision_tree', DecisionTreeRegressor(max_depth=5, random_state=42)),
    ('knn', KNeighborsRegressor(n_neighbors=5)),
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))
]

# Meta-model (final estimator)
meta_model = Ridge(alpha=1.0)

# Stacking Regressor
stacking_reg = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

print("Training Stacking Regressor...")
print("This may take a while as it trains multiple models with cross-validation...")
stacking_reg.fit(X_train, y_train)
y_pred_stacking = stacking_reg.predict(X_test)

# Evaluate
mae_stacking = mean_absolute_error(y_test, y_pred_stacking)
mse_stacking = mean_squared_error(y_test, y_pred_stacking)
rmse_stacking = np.sqrt(mse_stacking)
r2_stacking = r2_score(y_test, y_pred_stacking)

print("\nStacking Regressor Results:")
print(f"MAE: {mae_stacking:.4f}")
print(f"RMSE: {rmse_stacking:.4f}")
print(f"R²: {r2_stacking:.4f}")

In [ ]:
# Stacking with Random Forest as meta-model
stacking_reg_rf = StackingRegressor(
    estimators=base_estimators,
    final_estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    cv=5,
    n_jobs=-1
)

print("Training Stacking Regressor with Random Forest meta-model...")
stacking_reg_rf.fit(X_train, y_train)
y_pred_stacking_rf = stacking_reg_rf.predict(X_test)

# Evaluate
mae_stacking_rf = mean_absolute_error(y_test, y_pred_stacking_rf)
mse_stacking_rf = mean_squared_error(y_test, y_pred_stacking_rf)
rmse_stacking_rf = np.sqrt(mse_stacking_rf)
r2_stacking_rf = r2_score(y_test, y_pred_stacking_rf)

print("\nStacking (Random Forest meta-model) Results:")
print(f"MAE: {mae_stacking_rf:.4f}")
print(f"RMSE: {rmse_stacking_rf:.4f}")
print(f"R²: {r2_stacking_rf:.4f}")

## Step 7: Blending

**Blending Concept:**

Blending is similar to stacking but uses a simpler approach. Instead of using k-fold cross-validation to generate meta-features, blending uses a hold-out validation set.

**Blending Process:**

1. **Split Data**: Divide training data into training and validation sets
2. **Train Base Models**: Train each base model on the training set
3. **Generate Predictions**: Use each base model to predict on the validation set
4. **Create Meta-Features**: Use these predictions as features for the meta-model
5. **Train Meta-Model**: Train meta-model on the meta-features
6. **Final Prediction**: Use base models to predict on test data, then meta-model to combine

**Differences from Stacking:**
- **Simpler**: Uses a single hold-out set instead of k-fold CV
- **Faster**: Less computationally expensive
- **Less Data**: Uses less data for training base models (due to hold-out)
- **Simpler Implementation**: Easier to implement manually

**Advantages:**
- **Simple**: Easier to understand and implement
- **Fast**: Less computationally expensive than stacking
- **Effective**: Often achieves good performance

**Disadvantages:**
- **Less Data**: Uses less data for training (due to hold-out set)
- **Higher Variance**: More sensitive to the specific train/validation split
- **Less Robust**: Less robust than stacking with k-fold CV

In [ ]:
# Implement Blending manually

# Split training data into training and validation sets for blending
X_train_blend, X_val_blend, y_train_blend, y_val_blend = train_test_split(
    X_train,
    y_train,
    test_size=0.3,
    random_state=42
)

print(f"Blending Training Set: {X_train_blend.shape}")
print(f"Blending Validation Set: {X_val_blend.shape}")

# Train base models on training set
base_models = {
    'linear': LinearRegression(),
    'ridge': Ridge(alpha=1.0),
    'lasso': Lasso(alpha=1.0),
    'decision_tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'knn': KNeighborsRegressor(n_neighbors=5)
}

# Train base models and generate validation predictions
val_predictions = {}
for name, model in base_models.items():
    model.fit(X_train_blend, y_train_blend)
    val_predictions[name] = model.predict(X_val_blend)
    print(f"Trained {name}")

# Create meta-features DataFrame
meta_features_val = pd.DataFrame(val_predictions)

# Train meta-model on validation predictions
meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_features_val, y_val_blend)

print("\nTrained meta-model on validation predictions")

# Generate test predictions from base models
test_predictions = {}
for name, model in base_models.items():
    test_predictions[name] = model.predict(X_test)

# Create meta-features for test set
meta_features_test = pd.DataFrame(test_predictions)

# Final prediction using meta-model
y_pred_blending = meta_model.predict(meta_features_test)

# Evaluate
mae_blending = mean_absolute_error(y_test, y_pred_blending)
mse_blending = mean_squared_error(y_test, y_pred_blending)
rmse_blending = np.sqrt(mse_blending)
r2_blending = r2_score(y_test, y_pred_blending)

print("\nBlending Results:")
print(f"MAE: {mae_blending:.4f}")
print(f"RMSE: {rmse_blending:.4f}")
print(f"R²: {r2_blending:.4f}")

## Step 8: Compare All Ensemble Methods

Let's compare the performance of all ensemble methods we've implemented.

In [ ]:
# Collect results from all ensemble methods
ensemble_results = [
    {
        'Method': 'Bagging (Decision Tree)',
        'MAE': mae_bagging_dt,
        'RMSE': rmse_bagging_dt,
        'R²': r2_bagging_dt
    },
    {
        'Method': 'Bagging (Linear Regression)',
        'MAE': mae_bagging_lr,
        'RMSE': rmse_bagging_lr,
        'R²': r2_bagging_lr
    },
    {
        'Method': 'Voting Regressor',
        'MAE': mae_voting,
        'RMSE': rmse_voting,
        'R²': r2_voting
    },
    {
        'Method': 'Weighted Voting',
        'MAE': mae_voting_weighted,
        'RMSE': rmse_voting_weighted,
        'R²': r2_voting_weighted
    },
    {
        'Method': 'Stacking (Ridge meta)',
        'MAE': mae_stacking,
        'RMSE': rmse_stacking,
        'R²': r2_stacking
    },
    {
        'Method': 'Stacking (RF meta)',
        'MAE': mae_stacking_rf,
        'RMSE': rmse_stacking_rf,
        'R²': r2_stacking_rf
    },
    {
        'Method': 'Blending',
        'MAE': mae_blending,
        'RMSE': rmse_blending,
        'R²': r2_blending
    }
]

# Create DataFrame
ensemble_df = pd.DataFrame(ensemble_results)
ensemble_df = ensemble_df.sort_values('R²', ascending=False)

# Display results
print("Ensemble Methods Comparison (Sorted by R²):")
print("="*80)
display(ensemble_df.style.format({
    'MAE': '{:.4f}',
    'RMSE': '{:.4f}',
    'R²': '{:.4f}'
}))

## Step 9: Visualize Ensemble Performance

Let's create visualizations to compare the performance of all ensemble methods.

In [ ]:
# Create subplots for different metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# R² Score Comparison
colors_r2 = ['green' if x > 0.6 else 'orange' if x > 0.5 else 'red' for x in ensemble_df['R²']]
axes[0].barh(ensemble_df['Method'], ensemble_df['R²'], color=colors_r2)
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (0.5)')
axes[0].legend()
for i, v in enumerate(ensemble_df['R²']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# RMSE Comparison
axes[1].barh(ensemble_df['Method'], ensemble_df['RMSE'], color='steelblue')
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
for i, v in enumerate(ensemble_df['RMSE']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# MAE Comparison
axes[2].barh(ensemble_df['Method'], ensemble_df['MAE'], color='coral')
axes[2].set_xlabel('MAE')
axes[2].set_title('MAE Comparison', fontsize=14, fontweight='bold')
for i, v in enumerate(ensemble_df['MAE']):
    axes[2].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Step 10: Compare with Individual Base Models

Let's compare ensemble methods with individual base models to see the improvement.

In [ ]:
# Train individual base models for comparison
individual_results = []

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
individual_results.append({
    'Method': 'Linear Regression',
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
    'R²': r2_score(y_test, y_pred_lr)
})

# Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
individual_results.append({
    'Method': 'Ridge Regression',
    'MAE': mean_absolute_error(y_test, y_pred_ridge),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    'R²': r2_score(y_test, y_pred_ridge)
})

# Decision Tree
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
individual_results.append({
    'Method': 'Decision Tree',
    'MAE': mean_absolute_error(y_test, y_pred_dt),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_dt)),
    'R²': r2_score(y_test, y_pred_dt)
})

# KNN
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
individual_results.append({
    'Method': 'KNN',
    'MAE': mean_absolute_error(y_test, y_pred_knn),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_knn)),
    'R²': r2_score(y_test, y_pred_knn)
})

# Random Forest (as a reference ensemble)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
individual_results.append({
    'Method': 'Random Forest',
    'MAE': mean_absolute_error(y_test, y_pred_rf),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'R²': r2_score(y_test, y_pred_rf)
})

# Create DataFrame
individual_df = pd.DataFrame(individual_results)

# Combine with ensemble results
combined_df = pd.concat([individual_df, ensemble_df], ignore_index=True)
combined_df = combined_df.sort_values('R²', ascending=False)

# Display results
print("All Methods Comparison (Individual + Ensemble):")
print("="*80)
display(combined_df.style.format({
    'MAE': '{:.4f}',
    'RMSE': '{:.4f}',
    'R²': '{:.4f}'
}))

## Step 11: Visualize All Methods Comparison

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# R² Score Comparison
colors_r2 = ['green' if x > 0.6 else 'orange' if x > 0.5 else 'red' for x in combined_df['R²']]
axes[0].barh(combined_df['Method'], combined_df['R²'], color=colors_r2)
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score Comparison (All Methods)', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (0.5)')
axes[0].legend()
for i, v in enumerate(combined_df['R²']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)

# RMSE Comparison
axes[1].barh(combined_df['Method'], combined_df['RMSE'], color='steelblue')
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE Comparison (All Methods)', fontsize=14, fontweight='bold')
for i, v in enumerate(combined_df['RMSE']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)

# MAE Comparison
axes[2].barh(combined_df['Method'], combined_df['MAE'], color='coral')
axes[2].set_xlabel('MAE')
axes[2].set_title('MAE Comparison (All Methods)', fontsize=14, fontweight='bold')
for i, v in enumerate(combined_df['MAE']):
    axes[2].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## Step 12: Summary and Recommendations

Based on our comprehensive comparison, let's summarize the findings and provide recommendations.

In [ ]:
# Display final summary
print("="*80)
print("ENSEMBLE LEARNING COMPARISON SUMMARY")
print("="*80)
print()

print("\n📊 ENSEMBLE METHODS PERFORMANCE (by R² Score):")
print("-"*80)
for idx, row in ensemble_df.iterrows():
    print(f"{idx+1}. {row['Method']:<35} R²: {row['R²']:.4f} | RMSE: {row['RMSE']:.4f}")

print("\n🏆 BEST ENSEMBLE METHOD:")
print("-"*80)
best_ensemble = ensemble_df.iloc[0]
print(f"Method: {best_ensemble['Method']}")
print(f"R² Score: {best_ensemble['R²']:.4f} ({best_ensemble['R²']*100:.2f}% variance explained)")
print(f"RMSE: {best_ensemble['RMSE']:.4f}")
print(f"MAE: {best_ensemble['MAE']:.4f}")

print("\n📈 ENSEMBLE VS INDIVIDUAL MODELS:")
print("-"*80)
best_individual = individual_df.loc[individual_df['R²'].idxmax()]
print(f"Best Individual Model: {best_individual['Method']} (R²: {best_individual['R²']:.4f})")
print(f"Best Ensemble Method: {best_ensemble['Method']} (R²: {best_ensemble['R²']:.4f})")
improvement = (best_ensemble['R²'] - best_individual['R²']) * 100
print(f"\nImprovement: {improvement:.2f}%")

print("\n💡 KEY INSIGHTS:")
print("-"*80)
print("1. Ensemble methods generally outperform individual models")
print("2. Stacking often achieves the best performance")
print("3. Bagging with Decision Trees is effective and fast")
print("4. Voting provides good results with diverse models")
print("5. Blending is simpler than stacking but uses less data")

print("\n🎯 RECOMMENDATIONS:")
print("-"*80)
print("1. For maximum accuracy: Use Stacking with diverse base models")
print("2. For speed: Use Bagging or Voting")
print("3. For simplicity: Use Voting with pre-trained models")
print("4. For large datasets: Use Bagging (parallelizable)")
print("5. For small datasets: Use Stacking (better use of data)")
print("6. Always tune hyperparameters for optimal performance")

print("\n" + "="*80)

## Step 13: Ensemble Methods Characteristics Summary

Let's provide a summary of each ensemble method's characteristics.

In [ ]:
# Create ensemble characteristics summary
ensemble_characteristics = pd.DataFrame({
    'Method': [
        'Bagging',
        'Voting',
        'Stacking',
        'Blending'
    ],
    'Concept': [
        'Train multiple models on different data subsets',
        'Combine predictions from multiple models',
        'Use meta-model to combine base model predictions',
        'Similar to stacking but uses hold-out set'
    ],
    'Base Models': [
        'Same type (usually)',
        'Different types',
        'Different types',
        'Different types'
    ],
    'Parallelizable': [
        'Yes',
        'Yes',
        'No (sequential)',
        'Yes'
    ],
    'Training Speed': [
        'Fast (parallel)',
        'Medium',
        'Slow (sequential)',
        'Medium'
    ],
    'Data Usage': [
        'Full (with replacement)',
        'Full',
        'Full (with CV)',
        'Partial (hold-out)'
    ],
    'Complexity': [
        'Low',
        'Low',
        'High',
        'Medium'
    ],
    'Best For': [
        'High variance models, parallel processing',
        'Diverse models, simple combination',
        'Maximum accuracy, optimal combination',
        'Simpler alternative to stacking'
    ]
})

print("\nENSEMBLE METHODS CHARACTERISTICS:")
print("="*120)
display(ensemble_characteristics)
print("="*120)